# 🧭 Strategy Guide: Tackling Linear-Regression Projects on Survey/Observational Data
A general-purpose playbook — distilled from the PSID earnings project — for approaching
*any* new dataset with a linear regression lens: predicting or explaining a continuous
outcome from a mix of numeric and categorical predictors.

## The Big Picture: A 5-Phase Framework

```
PHASE 1: FRAME        -> What question, for whom, using what outcome?
PHASE 2: INTERROGATE   -> Understand the data before touching a model
PHASE 3: PREPARE       -> Clean, filter, transform, split
PHASE 4: MODEL         -> Simple -> assess -> multiple -> assess
PHASE 5: COMMUNICATE   -> Interpret honestly, state limits, summarize
```

Each phase below has a **checklist**, **decision guide**, and **red flags** to watch
for. This mirrors exactly what we did on the PSID project, generalized so you can reuse
it on a new dataset in a fraction of the time.

---
## Phase 1 — Frame the Question

Before opening the data, write down:

1. **The outcome variable** — what, precisely, are you trying to explain or predict?
2. **The population** — who/what does one row represent, and who should your
   conclusions apply to?
3. **The audience** — is this for a technical report, a business stakeholder, a
   policy question? This shapes how much statistical detail vs. plain-English framing
   you'll need at the end.
4. **A hypothesis or two** — e.g., "I expect more education to be associated with
   higher income" — so you have something concrete to confirm, refute, or refine.

> 🎯 **Rule of thumb:** if you can't state your question as one sentence with a named
> outcome variable and named predictor(s), you're not ready to load the data yet.

---
## Phase 2 — Interrogate the Data (before modeling anything)

### Checklist
- [ ] Load and check shape, dtypes (`.info()`), and summary stats (`.describe(include='all')`)
- [ ] For every numeric column, check `.unique()` / min / max for **suspicious sentinel
      values** (999, 99, -9, -99, 9999 are all common missing-data codes in survey data)
- [ ] For every categorical column, check `.value_counts(dropna=False)` — do the
      categories make sense? Is there a large "Refused"/"Don't know"/"NA" bucket?
- [ ] Check missingness patterns: is missingness random, or concentrated in a
      meaningful subgroup? (`groupby` + `.isna().mean()` is your friend)
- [ ] **For zero-inflated variables** (a huge spike at exactly 0): ask *why*. Structural
      zero (not applicable / not measured for this group) vs. genuine zero (truly zero)
      require completely different treatment.
- [ ] Read the codebook/documentation if one exists — never guess what a code means if
      you can look it up.

### Decision guide: "Is this missingness/zero problem structural or random?"
```
Is the "bad" value concentrated in one subgroup
(e.g., one employment status, one age range, one survey wave)?
        |
        +-- YES --> Likely STRUCTURAL. Consider restricting your analytic
        |           sample to the subgroup where the variable is actually
        |           measured, rather than "cleaning" it as noise.
        |
        +-- NO, scattered randomly --> Likely genuine missing-at-random.
                    Consider dropping rows, or imputing if you have a
                    principled method and can justify it.
```

### 🚩 Red flags to catch here
- A numeric max/min that's suspiciously round (99, 999, -1) → sentinel code
- A "continuous" variable that is 90%+ exactly zero → probably structural, not noise
- Category counts that don't sum to the total row count → hidden NaNs
- A column name that implies something the values don't support (e.g., a "years"
  column with a value of `999`)

---
## Phase 3 — Prepare the Data

### Checklist
- [ ] Replace all sentinel codes with `NaN` (do this **before** any stats or plots)
- [ ] Restrict to the population your question is actually about (e.g., working-age
      adults, not all household members of every age)
- [ ] Decide row-by-row inclusion rules and **write down why** for each one — this
      becomes your "Data Cleaning" section later
- [ ] Recode messy categoricals into a smaller number of meaningful, well-labeled groups
- [ ] Visualize the outcome's distribution. Right-skewed (income, price, wait times,
      counts)? Consider a log transform. Bounded 0/1 or a proportion? Consider whether
      linear regression is even the right tool (logistic regression may fit better)
- [ ] Check outliers **within your chosen population** using boxplots / IQR — only after
      structural issues are already handled
- [ ] Split into train/test **before** you start comparing model fit numbers, so your
      final evaluation is honest

### Decision guide: "Should I log-transform the outcome?"
```
Plot a histogram of the outcome.
        |
        +-- Roughly symmetric / bell-shaped -> No transform needed
        |
        +-- Long right tail (a few huge values pull the mean way above
        |   the median), and all values are strictly positive
        |         |
        |         +-- YES --> log-transform is usually a great, simple fix
        |
        +-- Contains zeros or negative values
                  |
                  +-- log(x + 1), or reconsider whether OLS is the
                      right model at all (e.g., Tobit/zero-inflated models
                      if zeros are a large, meaningful category)
```

### 🚩 Red flags to catch here
- Modeling a variable that is mostly zero as if it were continuous everywhere
- Filtering "outliers" using a rule computed on the *unfiltered, uncleaned* data
- Doing your train/test split *after* looking at test-set performance (data leakage /
  peeking)
- Recoding categories in a way that loses an important distinction your question needs

---
## Phase 4 — Model, Assess, Extend

### The core loop (repeat for each model)
```
1. FIT       -- smf.ols('y ~ x1 + x2 + ...', data=train).fit()
2. FIT STATS -- R², adjusted R², RSE / AIC / BIC
3. RESIDUALS -- residuals vs fitted, Q-Q plot -- do assumptions hold?
4. VISUALIZE -- plot the fit against the data (and a flexible LOWESS for comparison)
5. INTERPRET -- coefficients, p-values, CIs -- in the ORIGINAL units your audience cares about
6. PREDICT   -- evaluate on the held-out TEST set, compare to a naive baseline
```

### Decision guide: "Should I add this predictor?"
```
Does theory / domain knowledge suggest it plausibly affects the outcome?
        |
        +-- NO --> probably skip it (or note it as an idea for future work)
        |
        +-- YES --> add it, then check:
                    - Does adjusted R² improve (not just raw R²)?
                    - Is the nested F-test (anova_lm) significant?
                    - Do VIFs stay reasonable (no predictor > ~5-10)?
                    - Does the new predictor's own p-value / CI suggest
                      a real effect, not noise?
                          |
                          +-- All good --> keep it
                          +-- Fails checks --> reconsider; note as a
                              limitation rather than force it in
```

### Decision guide: "Simple vs. multiple regression — when am I done adding predictors?"
- Stop when adding a predictor no longer meaningfully improves **adjusted** R² or the
  nested F-test, or when you run out of variables theory says should matter.
- Watch for **diminishing, noisy returns**: a project with 900 rows and 15 predictors is
  probably overfit; keep the predictor count modest relative to sample size.
- If two predictors are highly correlated with each other (high VIF) — like `age` and
  `age²`, or `income` and `log(income)` — that's expected for engineered polynomial /
  transformed terms, but a red flag between two *separately measured* variables.

### 🚩 Red flags to catch here
- Reporting R² without ever looking at a residual plot
- Comparing raw R² across models with different numbers of predictors (use adjusted R²)
- Interpreting a categorical coefficient without naming the reference category
- Treating statistical significance (p < 0.05) as automatically meaning "practically
  important" — always look at the effect size too
- Evaluating a model only on the data it was trained on

---
## Phase 5 — Communicate Honestly

### Checklist
- [ ] State who your analytic sample represents — and who it does *not*
- [ ] Never claim causation from an observational regression coefficient — use language
      like "associated with," not "causes"
- [ ] Report effect sizes in plain, real-world units (dollars, percent, days) — not just
      standardized coefficients or p-values
- [ ] Include at least one honest limitation and one concrete next step
- [ ] Write an executive summary that a non-technical stakeholder could act on

### A reusable "Limitations" template
> "This analysis is based on [N] observations representing [population/sample
> definition]. Because the data is observational, the relationship between [predictor]
> and [outcome] should be read as an association, not a proven causal effect — factors
> like [plausible confounder] could influence both. [Known data quirk] limits how
> confidently we can interpret [specific variable]. A natural next step would be
> [longitudinal data / additional controls / a different model type] to strengthen these
> conclusions."

### 🚩 Red flags to catch here
- "Our model proves X causes Y" (regression alone never proves causation)
- Reporting only the best-looking metric (e.g., only training R², not test R²)
- Silently dropping caveats that would complicate a nice-sounding conclusion

---
## Quick-Start Template for a New Project

Copy this skeleton of questions into a markdown cell at the top of any new project and
answer each one before writing code:

```
1. Outcome variable: ____________________  (continuous? skewed? bounded?)
2. Unit of observation: ____________________  (person? household? firm? transaction?)
3. Target population for conclusions: ____________________
4. Candidate predictors (with expected sign of effect):
   - ____________________  (expect: + / - / unsure)
   - ____________________  (expect: + / - / unsure)
5. Known data quirks to check for (missing codes, structural zeros, top-coding): 
   - ____________________
6. What would a "good enough" model look like for this audience?
   (e.g., R² > 0.3, or "beats a naive baseline by 20%", or "coefficient sign matches theory")
```

Then walk the 5-phase framework above, using `04_reusable_template.ipynb` for the actual
code scaffolding (generic cleaning/modeling/diagnostic functions you can point at any new
dataset), and `03_project_cheatsheet.ipynb` for syntax lookups along the way.